## 1) Confounds: Non-chemically Meaningful Correlations

In this notebook, we test whether molecular properties are explained by simple SELFIES confounds (length/branch/ring/entropy) and whether latent `Z` captures chemistry beyond sequence artifacts.


In [ ]:
import json
import random
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, TensorDataset

import selfies as sf
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, QED

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

DATA_CSV = "../smiles_selfies_full.csv"
CKPT_PATH = "trained_models/final(H256-L512-linpool).pt"

MODEL_PY = Path("models/final_nat_linearpool.py")
if not MODEL_PY.exists():
    MODEL_PY = Path("transformer_vae/models/final_nat_linearpool.py")

OUTPUT_DIR = Path("../artifacts/confounds_step3")
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"
CONFOUND_DIR = OUTPUT_DIR / "confound_directions"

SEED = 42
BATCH_SIZE = 1024
RIDGE_ALPHAS = np.logspace(-3, 3, 13)

for d in [OUTPUT_DIR, PLOTS_DIR, TABLES_DIR, CONFOUND_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")
print(f"MODEL_PY={MODEL_PY}")


## 3) Loading Dataset



In [ ]:
df = pd.read_csv(DATA_CSV)
required_cols = {"smiles", "selfies"}
assert required_cols.issubset(df.columns), f"CSV must contain columns: {required_cols}"

df = df[["smiles", "selfies"]].copy()
df["tokens"] = df["selfies"].astype(str).apply(lambda x: list(sf.split_selfies(x)))

all_tokens = [tok for seq in df["tokens"] for tok in seq]
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
vocab = [PAD, SOS, EOS] + sorted(set(all_tokens))
vocab_size = len(vocab)

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for tok, idx in tok2id.items()}

PAD_ID = 0
SOS_ID = 1
EOS_ID = 2

def tokens_to_ids(tokens):
    return np.array([SOS_ID] + [tok2id[t] for t in tokens] + [EOS_ID], dtype=np.int64)

df["token_ids"] = df["tokens"].apply(tokens_to_ids)
sequences = df["token_ids"].tolist()

N = len(sequences)
max_len = max(len(seq) for seq in sequences)
data = np.zeros((N, max_len), dtype=np.int64)
for i, seq in enumerate(sequences):
    data[i, : len(seq)] = seq

indices = np.arange(N)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, shuffle=True)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, shuffle=True)

split = np.empty(N, dtype=object)
split[train_idx] = "train"
split[val_idx] = "val"
split[test_idx] = "test"
df["split"] = split

print(f"N={N}, vocab_size={vocab_size}, max_len={max_len}")
print(df["split"].value_counts())


## 4) Loading Model Checkpoint (Benchmark Pattern)


In [ ]:
assert MODEL_PY.exists(), f"Model file not found: {MODEL_PY}"

spec = importlib.util.spec_from_file_location("final_nat_linearpool", MODEL_PY)
final_nat_linearpool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(final_nat_linearpool)

load_final_model = final_nat_linearpool.load_final_model
FINAL_HIDDEN_SIZE = final_nat_linearpool.FINAL_HIDDEN_SIZE
FINAL_ATTENTION_HEADS = final_nat_linearpool.FINAL_ATTENTION_HEADS
FINAL_NUM_SLOTS = final_nat_linearpool.FINAL_NUM_SLOTS
FINAL_LAYERS = final_nat_linearpool.FINAL_LAYERS
FINAL_LATENT_SIZE = final_nat_linearpool.FINAL_LATENT_SIZE

model = load_final_model(
    vocab_size=vocab_size,
    max_len=max_len,
    ckpt_path=CKPT_PATH,
    device=device,
)
latent_size = FINAL_LATENT_SIZE

print("Model loaded")
print(f"hidden_size={FINAL_HIDDEN_SIZE}, latent_size={latent_size}, layers={FINAL_LAYERS}")


## 5) Encoding All Molecules to Latent `Z` (Optional if Z_mu.npy exists)


In [ ]:
input_ids = torch.from_numpy(data).long()
attention_mask = (input_ids != PAD_ID).long()
row_ids = torch.arange(N, dtype=torch.long)

dataset = TensorDataset(input_ids, attention_mask, row_ids)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

z_mu = np.zeros((N, latent_size), dtype=np.float32)
z_logvar = np.full((N, latent_size), np.nan, dtype=np.float32)
has_logvar = False

model.eval()
with torch.no_grad():
    for x_batch, attn_batch, idx_batch in loader:
        x_batch = x_batch.to(device)
        enc_out = model.encode(x_batch)

        if isinstance(enc_out, (tuple, list)) and len(enc_out) >= 2:
            mu, logvar = enc_out[0], enc_out[1]
            has_logvar = True
        else:
            mu, logvar = enc_out, None

        idx_np = idx_batch.cpu().numpy()
        z_mu[idx_np] = mu.detach().cpu().numpy().astype(np.float32)

        if logvar is not None:
            z_logvar[idx_np] = logvar.detach().cpu().numpy().astype(np.float32)

np.save(OUTPUT_DIR / "Z_mu.npy", z_mu)
if has_logvar:
    np.save(OUTPUT_DIR / "Z_logvar.npy", z_logvar)

print(f"Saved latent means: {OUTPUT_DIR / 'Z_mu.npy'}")
print(f"Z_mu shape = {z_mu.shape}")


## 6) Building Confound Panel `C` (Optional if panels_step3.csv exists)

- `selfies_len_tokens`
- `branch_token_count`
- `ring_token_count`
- `token_entropy` (Shannon entropy of token distribution)


In [ ]:
def shannon_entropy(tokens):
    if len(tokens) == 0:
        return 0.0
    counts = pd.Series(tokens).value_counts().to_numpy(dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum())

df["selfies_len_tokens"] = df["tokens"].apply(lambda toks: float(len(toks)))
df["branch_token_count"] = df["tokens"].apply(lambda toks: float(sum("Branch" in t for t in toks)))
df["ring_token_count"] = df["tokens"].apply(lambda toks: float(sum("Ring" in t for t in toks)))
df["token_entropy"] = df["tokens"].apply(shannon_entropy)

C_COLS = ["selfies_len_tokens", "branch_token_count", "ring_token_count", "token_entropy"]
print(df[C_COLS].describe().T)


## 7) Build Property Panel `Y` (RDKit)(Optional if panels_step3.csv exists)

Parse SMILES with RDKit and compute the requested Tier 1 + Tier 2 built-in descriptors.
Track molecule validity via `is_rdkit_valid` and keep invalid rows marked cleanly.


In [ ]:
Y_COLS = [
    "MolWt",
    "ExactMolWt",
    "HeavyAtomCount",
    "cLogP",
    "TPSA",
    "HBD",
    "HBA",
    "NumRotatableBonds",
    "RingCount",
    "AromaticRingCount",
    "FractionCSP3",
    "NumSpiroAtoms",
    "NumBridgeheadAtoms",
    "BertzCT",
    "QED",
]

for col in Y_COLS:
    df[col] = np.nan

def safe_float(fn):
    try:
        return float(fn())
    except Exception:
        return np.nan

is_valid = np.zeros(N, dtype=bool)

for i, smi in enumerate(df["smiles"].tolist()):
    if pd.isna(smi):
        continue
    try:
        mol = Chem.MolFromSmiles(str(smi))
    except Exception:
        mol = None

    if mol is None:
        continue

    is_valid[i] = True
    df.at[i, "MolWt"] = safe_float(lambda: Descriptors.MolWt(mol))
    df.at[i, "ExactMolWt"] = safe_float(lambda: Descriptors.ExactMolWt(mol))
    df.at[i, "HeavyAtomCount"] = safe_float(lambda: mol.GetNumHeavyAtoms())
    df.at[i, "cLogP"] = safe_float(lambda: Crippen.MolLogP(mol))
    df.at[i, "TPSA"] = safe_float(lambda: rdMolDescriptors.CalcTPSA(mol))
    df.at[i, "HBD"] = safe_float(lambda: rdMolDescriptors.CalcNumHBD(mol))
    df.at[i, "HBA"] = safe_float(lambda: rdMolDescriptors.CalcNumHBA(mol))
    df.at[i, "NumRotatableBonds"] = safe_float(lambda: rdMolDescriptors.CalcNumRotatableBonds(mol))
    df.at[i, "RingCount"] = safe_float(lambda: rdMolDescriptors.CalcNumRings(mol))
    df.at[i, "AromaticRingCount"] = safe_float(lambda: rdMolDescriptors.CalcNumAromaticRings(mol))
    df.at[i, "FractionCSP3"] = safe_float(lambda: rdMolDescriptors.CalcFractionCSP3(mol))
    df.at[i, "NumSpiroAtoms"] = safe_float(lambda: rdMolDescriptors.CalcNumSpiroAtoms(mol))
    df.at[i, "NumBridgeheadAtoms"] = safe_float(lambda: rdMolDescriptors.CalcNumBridgeheadAtoms(mol))
    df.at[i, "BertzCT"] = safe_float(lambda: Descriptors.BertzCT(mol))
    df.at[i, "QED"] = safe_float(lambda: QED.qed(mol))

df["is_rdkit_valid"] = is_valid
print(f"RDKit valid molecules: {int(is_valid.sum())}/{N} ({is_valid.mean():.4f})")


## 8) Save Full Panels Table (Optional if panels_step3.csv exists)

Save one joined panel containing identity fields, split labels, confounds `C`, properties `Y`, and RDKit validity flag.


In [ ]:
panel_cols = ["smiles", "selfies", "split"] + C_COLS + Y_COLS + ["is_rdkit_valid"]

panel_path_tables = TABLES_DIR / "panels_step3.csv"
panel_path_root = OUTPUT_DIR / "panels_step3.csv"

df.loc[:, panel_cols].to_csv(panel_path_tables, index=False)
df.loc[:, panel_cols].to_csv(panel_path_root, index=False)

print(f"Saved: {panel_path_tables}")
print(f"Saved: {panel_path_root}")


## 9) Correlations Between `Y` and `C` (optional if corr_pearson_YC.csv and corr_spearman_YC.csv exist)

- We Compute Pearson and Spearman correlation matrices (`Y` vs `C`)



In [ ]:
valid_df = df[df["is_rdkit_valid"]].copy()
yc = valid_df[Y_COLS + C_COLS].apply(pd.to_numeric, errors="coerce")

pearson_full = yc.corr(method="pearson")
spearman_full = yc.corr(method="spearman")

corr_pearson_yc = pearson_full.loc[Y_COLS, C_COLS]
corr_spearman_yc = spearman_full.loc[Y_COLS, C_COLS]

corr_pearson_yc.to_csv(TABLES_DIR / "corr_pearson_YC.csv")
corr_spearman_yc.to_csv(TABLES_DIR / "corr_spearman_YC.csv")

def save_corr_heatmap(matrix_df, title, out_path):
    plt.figure(figsize=(1.6 * len(C_COLS) + 2, 0.55 * len(Y_COLS) + 2))
    im = plt.imshow(matrix_df.values, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    plt.xticks(np.arange(len(C_COLS)), C_COLS, rotation=45, ha="right")
    plt.yticks(np.arange(len(Y_COLS)), Y_COLS)
    plt.title(title)
    cb = plt.colorbar(im)
    cb.set_label("Correlation")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()

save_corr_heatmap(corr_pearson_yc, "Pearson correlation: Y vs C", PLOTS_DIR / "heatmap_pearson_YC.png")
save_corr_heatmap(corr_spearman_yc, "Spearman correlation: Y vs C", PLOTS_DIR / "heatmap_spearman_YC.png")

rank_rows = []
for y in Y_COLS:
    for c in C_COLS:
        p = corr_pearson_yc.loc[y, c]
        s = corr_spearman_yc.loc[y, c]
        abs_corr = np.nanmax([abs(p), abs(s)])
        rank_rows.append(
            {
                "property": y,
                "confound": c,
                "abs_corr": abs_corr,
                "pearson_corr": p,
                "spearman_corr": s,
            }
        )

top20_confounded = pd.DataFrame(rank_rows).sort_values("abs_corr", ascending=False).head(20)
top20_confounded.to_csv(TABLES_DIR / "top20_confounded_pairs_YC.csv", index=False)

print(top20_confounded.head(10))


## 10) Comparing Explainability `Z->Y` vs `Z->C`

We fit on `train`, evaluate on `val/test` with standardized inputs/targets and `RidgeCV` over log-spaced alphas.
Report per-target `R^2` and aggregate comparisons to test whether confounds are easier to predict from latent space than chemical properties.


In [ ]:
split_arr = df["split"].to_numpy()
valid_mask = df["is_rdkit_valid"].to_numpy(dtype=bool)

train_mask = (split_arr == "train") & valid_mask
val_mask = (split_arr == "val") & valid_mask
test_mask = (split_arr == "test") & valid_mask

X_train = z_mu[train_mask]
X_val = z_mu[val_mask]
X_test = z_mu[test_mask]

def fit_ridge_single_target(X_train, y_train, X_val, y_val, X_test, y_test, alphas):
    y_train = np.asarray(y_train, dtype=float)
    y_val = np.asarray(y_val, dtype=float)
    y_test = np.asarray(y_test, dtype=float)

    train_finite = np.isfinite(y_train)
    if train_finite.sum() < 5:
        return {"r2_val": np.nan, "r2_test": np.nan, "alpha": np.nan, "coef": None}

    if np.std(y_train[train_finite]) < 1e-12:
        return {"r2_val": np.nan, "r2_test": np.nan, "alpha": np.nan, "coef": None}

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    Xtr_s = x_scaler.fit_transform(X_train[train_finite])
    ytr_s = y_scaler.fit_transform(y_train[train_finite].reshape(-1, 1)).ravel()

    ridge = RidgeCV(alphas=alphas)
    ridge.fit(Xtr_s, ytr_s)

    def score_split(X_split, y_split):
        split_finite = np.isfinite(y_split)
        if split_finite.sum() < 2:
            return np.nan
        if np.std(y_split[split_finite]) < 1e-12:
            return np.nan

        Xs = x_scaler.transform(X_split[split_finite])
        pred_s = ridge.predict(Xs)
        pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        return float(r2_score(y_split[split_finite], pred))

    safe_x_scale = np.where(x_scaler.scale_ == 0, 1.0, x_scaler.scale_)
    coef_original_x_space = (y_scaler.scale_[0] * ridge.coef_) / safe_x_scale

    return {
        "r2_val": score_split(X_val, y_val),
        "r2_test": score_split(X_test, y_test),
        "alpha": float(ridge.alpha_),
        "coef": coef_original_x_space.astype(np.float32),
    }

r2_y_rows = []
for y in Y_COLS:
    y_all = df[y].to_numpy(dtype=float)
    out = fit_ridge_single_target(
        X_train, y_all[train_mask],
        X_val, y_all[val_mask],
        X_test, y_all[test_mask],
        RIDGE_ALPHAS,
    )
    r2_y_rows.append(
        {
            "property": y,
            "r2_val": out["r2_val"],
            "r2_test": out["r2_test"],
            "alpha": out["alpha"],
        }
    )

r2_y_df = pd.DataFrame(r2_y_rows)
r2_y_df.to_csv(TABLES_DIR / "r2_Z_to_Y.csv", index=False)

r2_c_rows = []
confound_coef_map = {}
for c in C_COLS:
    c_all = df[c].to_numpy(dtype=float)
    out = fit_ridge_single_target(
        X_train, c_all[train_mask],
        X_val, c_all[val_mask],
        X_test, c_all[test_mask],
        RIDGE_ALPHAS,
    )
    r2_c_rows.append(
        {
            "confound": c,
            "r2_val": out["r2_val"],
            "r2_test": out["r2_test"],
            "alpha": out["alpha"],
        }
    )
    if out["coef"] is not None:
        confound_coef_map[c] = out["coef"]

r2_c_df = pd.DataFrame(r2_c_rows)
r2_c_df.to_csv(TABLES_DIR / "r2_Z_to_C.csv", index=False)

avg_y_val = float(np.nanmean(r2_y_df["r2_val"].to_numpy(dtype=float)))
avg_c_val = float(np.nanmean(r2_c_df["r2_val"].to_numpy(dtype=float)))
avg_y_test = float(np.nanmean(r2_y_df["r2_test"].to_numpy(dtype=float)))
avg_c_test = float(np.nanmean(r2_c_df["r2_test"].to_numpy(dtype=float)))

fig = plt.figure(figsize=(18, 6))

ax1 = fig.add_subplot(1, 2, 1)
group_labels = ["Z->Y", "Z->C"]
x = np.arange(len(group_labels))
w = 0.35
ax1.bar(x - w / 2, [avg_y_val, avg_c_val], width=w, label="val")
ax1.bar(x + w / 2, [avg_y_test, avg_c_test], width=w, label="test")
ax1.set_xticks(x)
ax1.set_xticklabels(group_labels)
ax1.set_ylabel("Average R^2")
ax1.set_title("Average explainability")
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2)
y_val_map = r2_y_df.set_index("property")["r2_val"].to_dict()
c_val_map = r2_c_df.set_index("confound")["r2_val"].to_dict()
target_labels = [f"Y:{p}" for p in Y_COLS] + [f"C:{c}" for c in C_COLS]
target_vals = [y_val_map[p] for p in Y_COLS] + [c_val_map[c] for c in C_COLS]
target_colors = ["tab:blue"] * len(Y_COLS) + ["tab:orange"] * len(C_COLS)
ax2.bar(np.arange(len(target_labels)), target_vals, color=target_colors)
ax2.axhline(0.0, color="black", linewidth=1.0)
ax2.set_xticks(np.arange(len(target_labels)))
ax2.set_xticklabels(target_labels, rotation=80, ha="right")
ax2.set_ylabel("Validation R^2")
ax2.set_title("Per-target validation R^2")

fig.tight_layout()
fig.savefig(PLOTS_DIR / "r2_Z_to_Y_vs_C.png", dpi=220)
plt.close(fig)

print(f"Average R^2 val: Z->Y={avg_y_val:.4f}, Z->C={avg_c_val:.4f}")
print(f"Average R^2 test: Z->Y={avg_y_test:.4f}, Z->C={avg_c_test:.4f}")


## 11) Residualizing `Y` Against Confounds `C`

First model `C->Y` on train and compute residual properties `Y_residual = Y - Y_hat(C)`.
Then re-run `Z->Y_residual` prediction and compare with original `Z->Y` to estimate how much explainability is driven by confounds.


In [ ]:
C_all = df[C_COLS].to_numpy(dtype=float)
Y_all = df[Y_COLS].to_numpy(dtype=float)

finite_c = np.isfinite(C_all).all(axis=1)
finite_y = np.isfinite(Y_all).all(axis=1)

cy_train_mask = train_mask & finite_c & finite_y
assert cy_train_mask.sum() > 10, "Not enough train rows with finite C and Y for residualization."

c_scaler = StandardScaler()
y_scaler = StandardScaler()

C_train_s = c_scaler.fit_transform(C_all[cy_train_mask])
Y_train_s = y_scaler.fit_transform(Y_all[cy_train_mask])

cy_model = RidgeCV(alphas=RIDGE_ALPHAS)
cy_model.fit(C_train_s, Y_train_s)

Y_hat = np.full_like(Y_all, np.nan, dtype=np.float64)
pred_mask = finite_c
Y_hat[pred_mask] = y_scaler.inverse_transform(cy_model.predict(c_scaler.transform(C_all[pred_mask])))

Y_resid = Y_all - Y_hat
resid_cols = []
for j, y in enumerate(Y_COLS):
    col = f"resid_{y}"
    df[col] = Y_resid[:, j]
    resid_cols.append(col)

orig_val_map = r2_y_df.set_index("property")["r2_val"].to_dict()
orig_test_map = r2_y_df.set_index("property")["r2_test"].to_dict()

r2_resid_rows = []
for y in Y_COLS:
    y_res = df[f"resid_{y}"].to_numpy(dtype=float)
    out = fit_ridge_single_target(
        X_train, y_res[train_mask],
        X_val, y_res[val_mask],
        X_test, y_res[test_mask],
        RIDGE_ALPHAS,
    )

    r2_orig_val = float(orig_val_map.get(y, np.nan))
    r2_orig_test = float(orig_test_map.get(y, np.nan))
    r2_resid_val = out["r2_val"]
    r2_resid_test = out["r2_test"]

    delta_val = np.nan
    if np.isfinite(r2_orig_val) and np.isfinite(r2_resid_val):
        delta_val = float(r2_resid_val - r2_orig_val)

    delta_test = np.nan
    if np.isfinite(r2_orig_test) and np.isfinite(r2_resid_test):
        delta_test = float(r2_resid_test - r2_orig_test)

    r2_resid_rows.append(
        {
            "property": y,
            "r2_original": r2_orig_val,
            "r2_residual": r2_resid_val,
            "delta": delta_val,
            "r2_original_test": r2_orig_test,
            "r2_residual_test": r2_resid_test,
            "delta_test": delta_test,
        }
    )

r2_resid_df = pd.DataFrame(r2_resid_rows)
r2_resid_df.to_csv(TABLES_DIR / "r2_Z_to_Y_vs_Yresid.csv", index=False)

x = np.arange(len(Y_COLS))
w = 0.38
delta_val = r2_resid_df.set_index("property").loc[Y_COLS, "delta"].to_numpy(dtype=float)
delta_test = r2_resid_df.set_index("property").loc[Y_COLS, "delta_test"].to_numpy(dtype=float)

plt.figure(figsize=(14, 6))
plt.bar(x - w / 2, delta_val, width=w, label="val")
plt.bar(x + w / 2, delta_test, width=w, label="test")
plt.axhline(0.0, color="black", linewidth=1.0)
plt.xticks(x, Y_COLS, rotation=80, ha="right")
plt.ylabel("Delta R^2 (residual - original)")
plt.title("Effect of residualizing Y against confounds C")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "delta_r2_Y_vs_Yresid.png", dpi=220)
plt.close()

panel_with_resid_cols = panel_cols + resid_cols
panel_resid_path_tables = TABLES_DIR / "panels_step3_with_residuals.csv"
panel_resid_path_root = OUTPUT_DIR / "panels_step3_with_residuals.csv"

df.loc[:, panel_with_resid_cols].to_csv(panel_resid_path_tables, index=False)
df.loc[:, panel_with_resid_cols].to_csv(panel_resid_path_root, index=False)

print(f"Saved: {TABLES_DIR / 'r2_Z_to_Y_vs_Yresid.csv'}")
print(f"Saved: {panel_resid_path_tables}")
print(f"Saved: {panel_resid_path_root}")


## 12) Confound Direction Vectors

We save latent-space coefficient vectors from `Z->confound` ridge models:
- `w_length`, `w_entropy`, `w_branch`, `w_ring`



In [ ]:
direction_files = {
    "selfies_len_tokens": "w_length.npy",
    "token_entropy": "w_entropy.npy",
    "branch_token_count": "w_branch.npy",
    "ring_token_count": "w_ring.npy",
}

saved_rows = []
for confound, filename in direction_files.items():
    coef = confound_coef_map.get(confound)
    if coef is None:
        continue
    out_path = CONFOUND_DIR / filename
    np.save(out_path, coef.astype(np.float32))
    saved_rows.append({"confound": confound, "path": str(out_path), "dim": int(coef.shape[0])})

directions_df = pd.DataFrame(saved_rows)
directions_df.to_csv(TABLES_DIR / "confound_directions_index.csv", index=False)

print(directions_df)
